In [ ]:
import numpy as np
import requests
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
from torch.nn import functional as F

import tiktoken 

In [3]:
# Equal-weighted average of the past

past_activation = torch.tensor([4, 1, -2, -3])
N = len(past_activation)

weights = torch.ones(N) / N # equal weights for this example

present_activation = sum(past_activation * weights)

print("Past:", past_activation)
print("Weights (importance):", weights)
print("Sum weights:", sum(weights))
print("Present (weighted sum of the past)", present_activation)

Past: tensor([ 4,  1, -2, -3])
Weights (importance): tensor([0.2500, 0.2500, 0.2500, 0.2500])
Sum weights: tensor(1.)
Present (weighted sum of the past) tensor(0.)


In [6]:
# try different weights
weights = torch.tensor([2, 1, 1, 1]) # dont add up 1

# option 1
linear_weights = weights / sum(weights)
present_linear = sum(past_activation * linear_weights)
print(linear_weights, sum(linear_weights))
print(present_linear)

print("-"*50)

# option 2
softmax_weights = torch.exp(weights) / sum(torch.exp(weights))
present_softmax = sum(past_activation * softmax_weights)
print(softmax_weights, sum(softmax_weights))
print(present_softmax)

tensor([0.4000, 0.2000, 0.2000, 0.2000]) tensor(1.)
tensor(0.8000)
--------------------------------------------------
tensor([0.4754, 0.1749, 0.1749, 0.1749]) tensor(1.)
tensor(1.2020)


In [9]:
# now let's do it longer, and our present won't be the end of the data, but something in the middle

activation_data = torch.tensor([4, 1, -2, -3, 8, 3, -1])
present_index = 4
N = len(activation_data)

weights = torch.ones(N)
weights[present_index+1:] = -torch.inf
weights

weights_linear = weights / torch.sum(weights)
weights_softmax = torch.exp(weights) / torch.sum(torch.exp(weights))

print(weights_linear)
print(weights_softmax)

tensor([-0., -0., -0., -0., -0., nan, nan])
tensor([0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000])


In [ ]:
# steps towards the future, for simplicity create weights equal to 1, in reality not all of the will be 1.
tril = torch.tril(torch.ones(9, 9))
tril[tril==0] = -torch.inf
tril_softmax = F.softmax(tril, dim=1) # always row-wise
tril_softmax

# the weights themselves are getting lower, since the probability must add 1, and it is divided between all the 1s in the row

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.0000],
        [0.1111, 0.1111, 0.1111, 0.1111, 0.1111, 0.1111, 0.1111, 0.1111, 0.1111]])

In [21]:
# Final demo with random activations
activations = torch.randn(N, N)
tril = torch.tril(torch.ones(N, N))
scaled_activations = activations * tril
scaled_activations[scaled_activations==0] = -torch.inf
softmax_past = F.softmax(scaled_activations, dim=1)

print("-activations\n", activations)

print("-past weight factor\n", tril)

print("-scaled past activations\n", scaled_activations)

print("-softmax past activations\n", softmax_past)

-activations
 tensor([[-0.2893,  0.2209, -1.1647, -1.4558,  2.4194,  0.0217, -0.4891],
        [ 0.5029,  0.3936, -0.7933, -0.5597, -0.1537,  0.8560,  0.3091],
        [ 0.9477, -0.1765, -0.4970, -0.0947,  2.0950, -0.2732, -0.2054],
        [ 0.3962,  0.2973,  1.5488, -0.8309,  1.6966,  1.2714, -0.5276],
        [ 0.6118,  0.3962,  1.3784, -0.0186, -0.9087, -1.1596,  1.0220],
        [ 0.7760, -1.3223,  1.6171,  0.3282,  0.5619, -2.2753,  0.6130],
        [-0.9718,  0.6786, -1.3398, -0.7833,  0.8735,  1.2981, -0.0316]])
-past weight factor
 tensor([[1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1.]])
-scaled past activations
 tensor([[-0.2893,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.5029,  0.3936,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.94

In [24]:
torch.sum(softmax_past, dim=1)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])